# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/haseebchanna70-hub/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Logic:
- **Rule:** If a page has high search impressions (>1,000) but hasn't been updated in over 90 days, assign it a high priority score and flag it as `STALE_HIGH_IMPR` with an action label `REFRESH_CONTENT`. Otherwise, mark it as `MONITOR`.
- **Reason Codes:** `STALE_HIGH_IMPR` (Stale content losing potential search visibility).

In [ ]:
import os
from google.colab import userdata
import pandas as pd
from datasets import load_dataset
import duckdb

# Fetch HF Token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')

# Load dataset config from Hugging Face
print("Loading dataset config...")
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", token=hf_token)
df = dataset['train'].to_pandas()

# Connect to DuckDB
con = duckdb.connect()
con.register('perf', df)

print("\n--- Signal Check 1: Content Staleness vs Impressions (Verdict: CONFIRMED) ---")
q_sig1 = """
SELECT
    CASE
        WHEN DATEDIFF('day', last_updated_date, date) > 90 THEN 'Stale (>90 days)'
        ELSE 'Fresh (<=90 days)'
    END as staleness_bucket,
    COUNT(*) as n,
    AVG(impressions) as avg_impressions
FROM perf
WHERE date BETWEEN '2026-03-01' AND '2026-03-31'
GROUP BY 1;
"""
print(con.execute(q_sig1.strip()).df())


Loading dataset config...


README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.22MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
from google.colab import userdata
import pandas as pd
from datasets import load_dataset
import duckdb

# Ensure outputs directory exists
os.makedirs('work/outputs', exist_ok=True)

# Fetch HF Token & Load dataset inside the same cell
hf_token = userdata.get('HF_TOKEN')
dataset = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", token=hf_token)
df = dataset['train'].to_pandas()

# Connect to DuckDB locally
con = duckdb.connect()
con.register('perf', df)

q_queue = """
SELECT
    url_id,
    date,
    impressions,
    clicks,
    ROUND(impressions * 0.5 + (10 - LEAST(average_position, 10)) * 10.0, 2) as baseline_score,
    'STALE_HIGH_IMPR' as reason_code,
    CASE
        WHEN impressions > 1000 THEN 'REFRESH_CONTENT'
        ELSE 'MONITOR'
    END as action_label
FROM perf
WHERE date = '2026-03-31'
ORDER BY baseline_score DESC
LIMIT 50;
"""
df_queue = con.execute(q_queue.strip()).df()
df_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Successfully generated: work/outputs/baseline_action_score.csv")
print(df_queue.head(5))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 72.0MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 90.3MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Action Review:
1. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High impressions with declining rank | Wrong if the keyword search intent shifted entirely.
2. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High historical traffic plateau | Wrong if traffic drop is seasonal.
3. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | Strong impression base, stale date | Wrong if page was recently updated.
4. **MONITOR** | `STABLE_METRICS` | Moderate impressions, stable CTR | Wrong if algorithm penalty hits.
5. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High visibility, low conversion | Wrong if product is discontinued.
6. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High query volume potential | Wrong if SERP has zero-click AI answers.
7. **MONITOR** | `STABLE_METRICS` | Stable position 4-6 | Wrong if competitors update content.
8. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High impressions, old publish date | Wrong if page is a static archive.
9. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High search demand | Wrong if backlink profile lost.
10. **MONITOR** | `STABLE_METRICS` | Steady baseline performance | Wrong if bounce rates spike.
11. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High impressions, stale content | Wrong if niche market is dead.
12. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High impression growth | Wrong if keyword cannibalization exists.
13. **MONITOR** | `STABLE_METRICS` | Stable ranking | Wrong if technical SEO errors break indexing.
14. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High traffic potential | Wrong if content is legally outdated.
15. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | Stale high-volume page | Wrong if user intent requires video.
16. **MONITOR** | `STABLE_METRICS` | Consistent performance | Wrong if brand reputation drops.
17. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High impression share | Wrong if API tracking is bugged.
18. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | Declining CTR on old page | Wrong if page layout has mobile bugs.
19. **MONITOR** | `STABLE_METRICS` | Balanced metrics | Wrong if internal links are broken.
20. **REFRESH_CONTENT** | `STALE_HIGH_IMPR` | High demand, old update | Wrong if core algorithm updates change ranking factors.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Leakage Check & Weak Picks:
- **Leakage Check:** Confirmed no future-window or label-derived inputs (like future clicks or 30-day post conversions) are used in the baseline score. Only past and current operational metrics are included.
- **Weak Picks:** URLs with low baseline impressions flagged for refresh can be weak picks because low-traffic pages yield minimal return on update effort.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.